# Leakage

**Prediction.** One row is one origin, one destination, and one departure month. The target is `OD_PAX`.

**Cutoff.** The start of that departure month, before that month's passengers have flown.

Anything created by the export, or measured from the same month's passengers, is not available at the cutoff. Airport attributes that do not change with the month are available.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
pax = pd.read_csv(root / "data" / "raw" / "AA_PAX_FORECASTING.csv")
air = pd.read_csv(root / "data" / "raw" / "AA_AIRPORT_INFO.csv")
print("forecasting columns:", list(pax.columns))
print("airport columns:", list(air.columns))


In [ ]:
index = pax["Unnamed: 0"].to_numpy()
print("Unnamed: 0 is 0..n-1 in file order:", bool((index == np.arange(len(pax))).all()))
print("SF_LOAD_TMS unique values, forecasting:", pax["SF_LOAD_TMS"].nunique(dropna=False))
print("SF_LOAD_TMS unique values, airports:", air["SF_LOAD_TMS"].nunique(dropna=False))
print(
    "Same-month OD_REV_USD correlation with OD_PAX:",
    round(pax["OD_REV_USD"].corr(pax["OD_PAX"]), 3),
)
print(
    "log1p correlation:",
    round(np.log1p(pax["OD_REV_USD"]).corr(np.log1p(pax["OD_PAX"])), 3),
)


`Unnamed: 0` is the export row number, 0 through 1,533,521, in file order. A future route-month does not have one.

`SF_LOAD_TMS` has one value in each file. It is the Snowflake load time of this extract, not a property of the departure month.

`OD_REV_USD` moves almost in lockstep with `OD_PAX` (correlation about 0.76, about 0.95 on the log scale). It is the revenue from those same passengers in that same month. It is not known at the cutoff. Revenue from earlier months would be allowed only as a lag, and this column is not a lag.

## Keep or drop

| Column | Role at the cutoff | Decision |
|---|---|---|
| `OD_PAX` | The target. Unknown at the cutoff. | Target, not a feature |
| `OD_REV_USD` | Same month's revenue from those passengers. | **Drop** |
| `Unnamed: 0` (both files) | Export row number. | **Drop** |
| `SF_LOAD_TMS` (both files) | One warehouse load timestamp for the whole extract. | **Drop** |
| `DEP_MONTH`, `DEP_YEAR` | The month being predicted. Known when the prediction is made. | **Keep** |
| `ORG_AIRPRT`, `DES_AIRPRT` | The route being predicted. | **Keep** |
| `AIRPRT_CD` | Join key onto the route's airports. Not a second feature after the join. | Join key |
| `AIRPRT_NM` | The name of `AIRPRT_CD`. Text, one per code. | **Drop** |
| `CITY_METRO_IATA_CD`, `CNTRY_CD`, `STATE_PROVNC_CD`, `WRLD_AREA_DOT_CD` | Fixed place attributes. State is blank for non-US airports; that blank is known before departure. | **Keep** |
| Latitude and longitude parts | Fixed location. Use them only after collapsing each into decimal degrees. | **Keep** as `LAT` and `LNG` |
| `LNGST_RUNWAY_FT_QTY`, `ELEVATN_FT_QTY` | Fixed airport attributes. | **Keep** |

No forecasting column other than the target and `OD_REV_USD` is an outcome of the month being predicted. The airport table does not contain passenger counts.

**Gradient-boosted trees**, because the columns worth keeping are a mix of route categories and fixed numeric airport attributes, and the leaky same-month revenue column has to stay out. A linear model would still be able to memorize `OD_REV_USD` if it were left in, which is the wrong reason for a high score.